# T02 Data Understanding

This notebook develops an academic-quality structural understanding of `weatherAUS.csv` before detailed EDA. It covers schema, variable groups, descriptive summaries, temporal coverage, target basics, and observed ranges. It does **not** perform missing-value analysis beyond basic counts, outlier treatment, preprocessing, feature engineering, data splitting, or modelling.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from fdm_rainfall.data import load_weather_data
from fdm_rainfall.understanding import (
    basic_range_checks,
    categorical_frequencies,
    categorical_summary,
    column_inventory,
    data_dictionary,
    dataset_overview,
    date_understanding,
    feature_group_table,
    numerical_summary,
    target_summary,
    write_data_dictionary,
)

DATASET_PATH = PROJECT_ROOT / 'data' / 'raw' / 'weatherAUS.csv'
TABLES_DIR = PROJECT_ROOT / 'reports' / 'tables'
DICTIONARY_PATH = PROJECT_ROOT / 'docs' / 'data_dictionary.md'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')

Project root: D:\11.Running Projects\FDM Mini Project\FDM_Mini_Project


## 1. Dataset overview

The dataset contains daily observations from Australian weather-station locations. Its project purpose is to support prediction of the binary `RainTomorrow` target and downstream weather-risk decision support. Feature definitions and units are documented from the Rattle weatherAUS reference and Australian Bureau of Meteorology Daily Weather Observations pages.

In [2]:
weather = load_weather_data(DATASET_PATH)
overview_table = dataset_overview(weather)
print(f'Loaded successfully: {weather.shape[0]:,} rows x {weather.shape[1]} columns')
display(overview_table)

Loaded successfully: 145,460 rows x 23 columns


,Characteristic,Value
0,Dataset purpose,Daily Australian weather observations for next...
1,Rows,145460
2,Columns,23
3,Minimum date,2007-11-01
4,Maximum date,2017-06-25
5,Locations,49
6,Target variable,RainTomorrow


## 2. Complete column inventory

Unique counts exclude missing values. Missing percentages are descriptive only; detailed missing-value analysis is deferred to T03.

In [3]:
column_inventory_table = column_inventory(weather)
display(column_inventory_table)

,Column,Pandas dtype,Logical variable type,Non-null values,Missing values,Missing percentage,Unique non-null values
0,Date,str,date/time,145460,0,0.0000,3436
1,Location,str,categorical nominal,145460,0,0.0000,49
2,MinTemp,float64,numerical continuous,143975,1485,1.0209,389
3,MaxTemp,float64,numerical continuous,144199,1261,0.8669,505
4,Rainfall,float64,numerical continuous,142199,3261,2.2419,681
5,Evaporation,float64,numerical continuous,82670,62790,43.1665,358
6,Sunshine,float64,numerical continuous,75625,69835,48.0098,145
7,WindGustDir,str,categorical nominal,135134,10326,7.0989,16
8,WindGustSpeed,float64,numerical continuous,135197,10263,7.0555,67
9,WindDir9am,str,categorical nominal,134894,10566,7.2639,16


## 3. Feature grouping

The groups below organize related measurements without transforming them.

In [4]:
feature_groups_table = feature_group_table()
display(feature_groups_table.groupby('Feature group', sort=False)['Feature'].apply(lambda values: ', '.join(values)).reset_index())

,Feature group,Feature
0,Date / Location,"Date, Location"
1,Temperature,"MinTemp, MaxTemp, Temp9am, Temp3pm"
2,Rain / evaporation / sunshine,"Rainfall, Evaporation, Sunshine, RainToday"
3,Wind,"WindGustDir, WindGustSpeed, WindDir9am, WindDi..."
4,Humidity,"Humidity9am, Humidity3pm"
5,Pressure,"Pressure9am, Pressure3pm"
6,Cloud,"Cloud9am, Cloud3pm"
7,Target,RainTomorrow


## 4. Numerical feature summary

These are unmodified descriptive statistics. No outlier interpretation or treatment is performed here.

In [5]:
numeric_summary_table = numerical_summary(weather)
display(numeric_summary_table.round(4))

,Feature,Count,Mean,Standard deviation,Minimum,25th percentile,Median,75th percentile,Maximum
0,MinTemp,143975.0,12.1940,6.3985,-8.5,7.6,12.0,16.9,33.9
1,MaxTemp,144199.0,23.2213,7.1190,-4.8,17.9,22.6,28.2,48.1
2,Rainfall,142199.0,2.3609,8.4781,0.0,0.0,0.0,0.8,371.0
3,Evaporation,82670.0,5.4682,4.1937,0.0,2.6,4.8,7.4,145.0
4,Sunshine,75625.0,7.6112,3.7855,0.0,4.8,8.4,10.6,14.5
5,WindGustSpeed,135197.0,40.0352,13.6071,6.0,31.0,39.0,48.0,135.0
6,WindSpeed9am,143693.0,14.0434,8.9154,0.0,7.0,13.0,19.0,130.0
7,WindSpeed3pm,142398.0,18.6627,8.8098,0.0,13.0,19.0,24.0,87.0
8,Humidity9am,142806.0,68.8808,19.0292,0.0,57.0,70.0,83.0,100.0
9,Humidity3pm,140953.0,51.5391,20.7959,0.0,37.0,52.0,66.0,100.0


## 5. Categorical feature summary

The compact summary gives cardinality, values, modes, and missing counts. `Location` is kept readable by showing only its ten most frequent values below; all 49 location frequencies are saved in the long-form output table. The remaining category frequencies are shown as a compact pivot.

In [6]:
categorical_summary_table = categorical_summary(weather)
categorical_frequencies_table = categorical_frequencies(weather)
display(categorical_summary_table)

print('Ten most frequent Location values:')
display(categorical_frequencies_table.query("Feature == 'Location'").head(10))

print('Frequencies for non-Location categorical values:')
display(
    categorical_frequencies_table.query("Feature != 'Location'")
    .pivot(index='Category', columns='Feature', values='Count')
    .fillna('')
)

,Feature,Number of categories,Category values,Most frequent category,Most frequent count,Missing count
0,Location,49,See 02_categorical_frequencies.csv for 49 loca...,Canberra,3436,0
1,WindGustDir,16,E | ENE | ESE | N | NE | NNE | NNW | NW | S | ...,W,9915,10326
2,WindDir9am,16,E | ENE | ESE | N | NE | NNE | NNW | NW | S | ...,N,11758,10566
3,WindDir3pm,16,E | ENE | ESE | N | NE | NNE | NNW | NW | S | ...,SE,10838,4228
4,RainToday,2,No | Yes,No,110319,3261
5,RainTomorrow,2,No | Yes,No,110316,3267


Ten most frequent Location values:


,Feature,Category,Count,Percentage of non-null
0,Location,Canberra,3436,2.3622
1,Location,Sydney,3344,2.2989
2,Location,Melbourne,3193,2.1951
3,Location,Brisbane,3193,2.1951
4,Location,Adelaide,3193,2.1951
5,Location,Perth,3193,2.1951
6,Location,Hobart,3193,2.1951
7,Location,Darwin,3193,2.1951
8,Location,Albury,3040,2.0899
9,Location,Wollongong,3040,2.0899


Frequencies for non-Location categorical values:


Feature,RainToday,RainTomorrow,WindDir3pm,WindDir9am,WindGustDir
Category,,,,,
E,,,8472.0,9176.0,9181.0
ENE,,,7857.0,7836.0,8104.0
ESE,,,8505.0,7630.0,7372.0
N,,,8890.0,11758.0,9313.0
NE,,,8263.0,7671.0,7133.0
NNE,,,6590.0,8129.0,6548.0
NNW,,,7870.0,7980.0,6620.0
NW,,,8610.0,8749.0,8122.0
No,110319.0,110316.0,,,


## 6. Date understanding

Dates are parsed safely in a separate analysis series; the raw `Date` column is not replaced. Global ordering and within-location ordering are assessed, but the data is not split.

In [7]:
date_result = date_understanding(weather)
date_coverage_summary_table = date_result.overview
year_counts_table = date_result.year_counts
location_date_coverage_table = date_result.location_coverage
display(date_coverage_summary_table)
display(year_counts_table)
print('Location-level coverage (first 10 of 49; complete table saved below):')
display(location_date_coverage_table.head(10))

,Date characteristic,Value
0,Minimum date,2007-11-01
1,Maximum date,2017-06-25
2,Calendar years represented,11
3,Elapsed coverage in years,9.6484
4,Invalid or missing dates,0
5,Raw file globally date-ordered,False
6,Locations internally date-ordered,49 of 49
7,All locations have the same start/end coverage,False
8,Distinct location coverage patterns,7


,Year,Record count
0,2007,61
1,2008,2270
2,2009,16789
3,2010,16782
4,2011,15407
5,2012,15409
6,2013,16415
7,2014,17885
8,2015,17885
9,2016,17934


Location-level coverage (first 10 of 49; complete table saved below):


,Location,Start date,End date,Record count,Unique dates
0,Adelaide,2008-07-01,2017-06-25,3193,3193
1,Albany,2008-12-01,2017-06-25,3040,3040
2,Albury,2008-12-01,2017-06-25,3040,3040
3,AliceSprings,2008-12-01,2017-06-25,3040,3040
4,BadgerysCreek,2009-01-01,2017-06-25,3009,3009
5,Ballarat,2008-12-01,2017-06-25,3040,3040
6,Bendigo,2008-12-01,2017-06-25,3040,3040
7,Brisbane,2008-07-01,2017-06-25,3193,3193
8,Cairns,2008-12-01,2017-06-25,3040,3040
9,Canberra,2007-11-01,2017-06-25,3436,3436


## 7. Target understanding

Only target type, classes, counts, and completeness are reported. Deeper class-imbalance analysis is deferred.

In [8]:
target_summary_table = target_summary(weather)
display(target_summary_table)

,Target characteristic,Value
0,Pandas dtype,str
1,Classes,No | Yes
2,No count,110316
3,Yes count,31877
4,Missing count,3267
5,Labelled count,142193


## 8. Basic observed ranges

The table reports observed endpoints and conservative flags for later investigation. Flagging does not mean that a value is invalid or should be removed. Detailed outlier and suspicious-value analysis belongs to T05.

In [9]:
range_checks_table = basic_range_checks(weather)
display(range_checks_table)
print('Ranges explicitly flagged for later investigation:')
display(range_checks_table[~range_checks_table['Later-investigation flag'].str.startswith('No basic')])

,Feature group,Feature,Unit / scale,Non-null count,Missing count,Observed minimum,Observed maximum,Later-investigation flag
0,Temperature,MinTemp,degrees Celsius (°C),143975,1485,-8.5,33.9,No basic range flag at T02; detailed assessmen...
1,Temperature,MaxTemp,degrees Celsius (°C),144199,1261,-4.8,48.1,No basic range flag at T02; detailed assessmen...
2,Temperature,Temp9am,degrees Celsius (°C),143693,1767,-7.2,40.2,No basic range flag at T02; detailed assessmen...
3,Temperature,Temp3pm,degrees Celsius (°C),141851,3609,-5.4,46.7,No basic range flag at T02; detailed assessmen...
4,Humidity,Humidity9am,percent (%),142806,2654,0.0,100.0,A 0% boundary observation occurs; verify conte...
5,Humidity,Humidity3pm,percent (%),140953,4507,0.0,100.0,A 0% boundary observation occurs; verify conte...
6,Pressure,Pressure9am,"hectopascals (hPa), mean-sea-level pressure",130395,15065,980.5,1041.0,No basic range flag at T02; detailed assessmen...
7,Pressure,Pressure3pm,"hectopascals (hPa), mean-sea-level pressure",130432,15028,977.1,1039.6,No basic range flag at T02; detailed assessmen...
8,Wind speed,WindGustSpeed,kilometres per hour (km/h),135197,10263,6.0,135.0,High wind-speed maximum; retain and investigat...
9,Wind speed,WindSpeed9am,kilometres per hour (km/h),143693,1767,0.0,130.0,High wind-speed maximum; retain and investigat...


Ranges explicitly flagged for later investigation:


,Feature group,Feature,Unit / scale,Non-null count,Missing count,Observed minimum,Observed maximum,Later-investigation flag
4,Humidity,Humidity9am,percent (%),142806,2654,0.0,100.0,A 0% boundary observation occurs; verify conte...
5,Humidity,Humidity3pm,percent (%),140953,4507,0.0,100.0,A 0% boundary observation occurs; verify conte...
8,Wind speed,WindGustSpeed,kilometres per hour (km/h),135197,10263,6.0,135.0,High wind-speed maximum; retain and investigat...
9,Wind speed,WindSpeed9am,kilometres per hour (km/h),143693,1767,0.0,130.0,High wind-speed maximum; retain and investigat...
11,Rainfall,Rainfall,millimetres (mm),142199,3261,0.0,371.0,Very high daily maximum; retain and investigat...
12,Evaporation,Evaporation,"millimetres (mm), Class A pan evaporation",82670,62790,0.0,145.0,Very high recorded maximum; retain and investi...
14,Cloud,Cloud9am,oktas (eighths of sky covered; documented scal...,89572,55888,0.0,9.0,Maximum exceeds the documented 0–8 clear-to-ov...
15,Cloud,Cloud3pm,oktas (eighths of sky covered; documented scal...,86102,59358,0.0,9.0,Maximum exceeds the documented 0–8 clear-to-ov...


## 9. Data dictionary

The complete dictionary uses documented feature meanings and units, actual local-file data types and quality notes, and explicitly deferred preprocessing considerations.

In [10]:
data_dictionary_table = data_dictionary(weather)
write_data_dictionary(DICTIONARY_PATH, weather)
display(data_dictionary_table)
print(f'Wrote complete dictionary to {DICTIONARY_PATH.relative_to(PROJECT_ROOT)}')

,Feature name,Description,Data type,Logical type,Unit / category meaning,Role,Basic data-quality note,Possible preprocessing consideration
0,Date,Calendar date of the daily weather observation.,str,date/time,YYYY-MM-DD,temporal,"0 missing (0.0000%); 3,436 unique non-null val...",Parse safely as datetime; preserve chronology ...
1,Location,Common name of the Australian weather-station ...,str,categorical nominal,Australian weather-station location name,location,0 missing (0.0000%); 49 unique non-null values.,Treat as nominal; encoding and location-aware ...
2,MinTemp,Minimum temperature recorded for the day.,float64,numerical continuous,degrees Celsius (°C),predictor,"1,485 missing (1.0209%); 389 unique non-null v...",Retain numeric values; missing-value and scali...
3,MaxTemp,Maximum temperature recorded for the day.,float64,numerical continuous,degrees Celsius (°C),predictor,"1,261 missing (0.8669%); 505 unique non-null v...",Retain numeric values; missing-value and scali...
4,Rainfall,Rainfall recorded for the day.,float64,numerical continuous,millimetres (mm),predictor,"3,261 missing (2.2419%); 681 unique non-null v...",Retain numeric values; missing-value and scali...
5,Evaporation,Class A pan evaporation in the 24 hours to 9am.,float64,numerical continuous,"millimetres (mm), Class A pan evaporation",predictor,"62,790 missing (43.1665%); 358 unique non-null...",Retain numeric values; missing-value and scali...
6,Sunshine,Duration of bright sunshine during the day.,float64,numerical continuous,hours,predictor,"69,835 missing (48.0098%); 145 unique non-null...",Retain numeric values; missing-value and scali...
7,WindGustDir,Direction of the strongest wind gust in the 24...,str,categorical nominal,compass direction category,predictor,"10,326 missing (7.0989%); 16 unique non-null v...",Preserve categories; missing-value handling an...
8,WindGustSpeed,Speed of the strongest wind gust in the 24 hou...,float64,numerical continuous,kilometres per hour (km/h),predictor,"10,263 missing (7.0555%); 67 unique non-null v...",Retain numeric values; missing-value and scali...
9,WindDir9am,Wind direction at the 9am observation.,str,categorical nominal,compass direction category,predictor,"10,566 missing (7.2639%); 16 unique non-null v...",Preserve categories; missing-value handling an...


Wrote complete dictionary to docs\data_dictionary.md


## 10. Save reusable summary tables

Only derived T02 summaries are written. The raw CSV is not modified.

In [11]:
tables = {
    '02_dataset_overview.csv': overview_table,
    '02_column_inventory.csv': column_inventory_table,
    '02_feature_groups.csv': feature_groups_table,
    '02_numeric_summary.csv': numeric_summary_table,
    '02_categorical_summary.csv': categorical_summary_table,
    '02_categorical_frequencies.csv': categorical_frequencies_table,
    '02_date_coverage_summary.csv': date_coverage_summary_table,
    '02_year_counts.csv': year_counts_table,
    '02_location_date_coverage.csv': location_date_coverage_table,
    '02_target_summary.csv': target_summary_table,
    '02_range_checks.csv': range_checks_table,
}
for filename, table in tables.items():
    table.to_csv(TABLES_DIR / filename, index=False)
print(f'Saved {len(tables)} T02 tables to {TABLES_DIR.relative_to(PROJECT_ROOT)}')

Saved 11 T02 tables to reports\tables


## T02 boundary

Data understanding ends here. Missing-value analysis (T03), relationship EDA, outlier decisions, leakage/splitting, preprocessing, feature engineering, and modelling have not started.